#  Wearable Stress Detection Using Physiological Signals
**Dataset:** WESAD (Wearable Stress and Affect Detection) — 15 subjects  
**Method:** Signal processing → Feature extraction → Machine Learning  
**Result:** 96.5%+ accuracy across 5 subjects

In [ ]:
# Sensör kanallarını göster
print("=== GÖĞÜS CİHAZI SENSÖRLERİ ===")
print(data['signal']['chest'].keys())

print("\n=== BİLEK CİHAZI SENSÖRLERİ ===")
print(data['signal']['wrist'].keys())

# Label dağılımı
import numpy as np
labels, counts = np.unique(data['label'], return_counts=True)
label_names = {0:'Geçiş', 1:'Baseline', 2:'Stres', 3:'Amusement', 4:'Meditasyon'}
print("\n=== LABEL DAĞILIMI ===")
for l, c in zip(labels, counts):
    if l in label_names:
        print(f"{label_names[l]:15s}: {c:6d} örnek")

##  1. Data Loading
WESAD dataset is stored in `.pkl` format with one file per subject.
Data was collected from two devices simultaneously:

- **Chest device:** ECG, EDA, EMG, Temp, Resp (700 Hz)
- **Wrist device:** BVP, EDA, TEMP, ACC (64 Hz)

Labels: 0=Transient, 1=Baseline, 2=Stress, 3=Amusement, 4=Meditation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sadece stres ve baseline kullanalım (2 ve 1)
label_names = {1:'Baseline', 2:'Stres', 3:'Amusement', 4:'Meditasyon'}
counts = {1: 800800, 2: 430500, 3: 253400, 4: 537599}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('WESAD Dataset — S2 Katılımcısı Genel Bakış', fontsize=14, fontweight='bold')

# 1. Label dağılımı (bar chart)
ax1 = axes[0]
colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']
bars = ax1.bar(label_names.values(), counts.values(), color=colors, edgecolor='white', linewidth=1.5)
ax1.set_title('Label Dağılımı')
ax1.set_ylabel('Örnek Sayısı')
for bar, val in zip(bars, counts.values()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
             f'{val:,}', ha='center', va='bottom', fontsize=9)

# 2. ECG sinyali örneği (stres vs baseline)
ax2 = axes[1]
ecg = data['signal']['chest']['ECG'].flatten()
labels = data['label']

# Stres anından 1000 örnek
stress_idx = np.where(labels == 2)[0][:1000]
baseline_idx = np.where(labels == 1)[0][:1000]

ax2.plot(ecg[baseline_idx], color='#2196F3', alpha=0.7, label='Baseline', linewidth=0.8)
ax2.plot(ecg[stress_idx], color='#F44336', alpha=0.7, label='Stres', linewidth=0.8)
ax2.set_title('ECG Sinyali: Stres vs Baseline')
ax2.set_xlabel('Örnek')
ax2.set_ylabel('Amplitüd')
ax2.legend()

plt.tight_layout()
plt.savefig('wesad_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Görsel kaydedildi!")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 15))
fig.suptitle('WESAD — All Chest Sensors: Stress vs Baseline', fontsize=16, fontweight='bold')

sensors = ['ECG', 'EDA', 'EMG', 'Temp', 'Resp']
colors_b = '#2196F3'
colors_s = '#F44336'

labels = data['label']
N = 5000  # daha uzun pencere

stress_idx = np.where(labels == 2)[0][:N]
baseline_idx = np.where(labels == 1)[0][:N]
time = np.arange(N) / 700

sensor_descriptions = {
    'ECG': 'Electrocardiogram (Heart Activity)',
    'EDA': 'Electrodermal Activity (Skin Conductance)',
    'EMG': 'Electromyogram (Muscle Activity)',
    'Temp': 'Body Temperature (°C)',
    'Resp': 'Respiration (Breathing Pattern)'
}

for i, sensor in enumerate(sensors):
    ax = axes[i//2][i%2]
    signal = data['signal']['chest'][sensor].flatten()
    ax.plot(time, signal[baseline_idx], color=colors_b, alpha=0.8, label='Baseline', linewidth=0.9)
    ax.plot(time, signal[stress_idx], color=colors_s, alpha=0.8, label='Stress', linewidth=0.9)
    ax.set_title(sensor_descriptions[sensor], fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (seconds)', fontsize=10)
    ax.set_ylabel('Amplitude', fontsize=10)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

# EDA Wrist — ayrı örnekleme hızı (64 Hz)
ax = axes[2][1]
eda_wrist = data['signal']['wrist']['EDA'].flatten()
N_w = min(320, len(eda_wrist))  # 64Hz × 5sn
time_w = np.arange(N_w) / 64

# Wrist için ayrı label array oluştur (downsample)
wrist_labels = data['label'][::11][:len(eda_wrist)]  # 700/64 ≈ 11
stress_w = np.where(wrist_labels == 2)[0][:N_w]
baseline_w = np.where(wrist_labels == 1)[0][:N_w]
min_len = min(len(stress_w), len(baseline_w), N_w)

ax.plot(np.arange(min_len)/64, eda_wrist[baseline_w[:min_len]], color=colors_b, alpha=0.8, label='Baseline', linewidth=1.2)
ax.plot(np.arange(min_len)/64, eda_wrist[stress_w[:min_len]], color=colors_s, alpha=0.8, label='Stress', linewidth=1.2)
ax.set_title('EDA — Wrist Device', fontsize=12, fontweight='bold')
ax.set_xlabel('Time (seconds)', fontsize=10)
ax.set_ylabel('Amplitude', fontsize=10)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('wesad_all_sensors.png', dpi=150, bbox_inches='tight')
plt.show()

##  Sensor Analysis Findings

| Sensor | Usefulness for Stress Detection | Observation |
|--------|--------------------------------|-------------|
| ECG |  Medium (0.173) | Heart rhythm changes under stress |
| EDA |  High (0.336) | Skin conductance is the strongest stress indicator |
| EMG |  Low (0.110) | High inter-subject variability, weak discriminator |
| Temp |  Lowest (0.065) | Slow-changing signal, limited short-term information |
| Resp |  High (0.315) | Breathing pattern changes significantly under stress |

**Key Finding:** EDA and Respiration are the two strongest predictors of stress,
consistent with physiological stress response theory.

## 2. Exploratory Data Analysis (EDA)
We visualize each sensor signal during stress and baseline conditions
to understand which sensors best differentiate stress from rest.

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal as scipy_signal
from scipy.stats import skew, kurtosis

# Örnekleme frekansları
FS_CHEST = 700  # Hz
WINDOW_SEC = 60  # 60 saniyelik pencere
STEP_SEC = 30    # 30 saniye kaydırma (overlap)

WINDOW_SIZE = FS_CHEST * WINDOW_SEC
STEP_SIZE = FS_CHEST * STEP_SEC

def extract_features(sig, label_window):
    """Bir pencereden istatistiksel özellikler çıkar"""
    features = {}
    
    # Temel istatistikler
    features['mean'] = np.mean(sig)
    features['std'] = np.std(sig)
    features['min'] = np.min(sig)
    features['max'] = np.max(sig)
    features['range'] = np.max(sig) - np.min(sig)
    features['skew'] = skew(sig)
    features['kurtosis'] = kurtosis(sig)
    
    # Label: penceredeki en sık label
    features['label'] = np.bincount(label_window.astype(int)).argmax()
    
    return features

# Tüm sensörler için feature extraction
sensors = ['ECG', 'EDA', 'EMG', 'Temp', 'Resp']
labels = data['label']

all_features = []

for start in range(0, len(labels) - WINDOW_SIZE, STEP_SIZE):
    end = start + WINDOW_SIZE
    label_window = labels[start:end]
    
    # Sadece saf stres (2) veya baseline (1) pencereleri al
    unique = np.unique(label_window)
    if len(unique) != 1 or unique[0] not in [1, 2]:
        continue
    
    window_features = {}
    for sensor in sensors:
        sig = data['signal']['chest'][sensor].flatten()[start:end]
        feats = extract_features(sig, label_window)
        for k, v in feats.items():
            if k != 'label':
                window_features[f'{sensor}_{k}'] = v
    
    window_features['label'] = unique[0]
    all_features.append(window_features)

df = pd.DataFrame(all_features)
print(f" Toplam pencere sayısı: {len(df)}")
print(f" Feature sayısı: {len(df.columns)-1}")
print(f"\nLabel dağılımı:\n{df['label'].value_counts()}")
print(f"\nİlk 3 satır:\n{df.head(3)}")

##  3. Feature Extraction
We extract statistical features from raw sensor signals using a sliding window approach.

- **Window size:** 60 seconds
- **Step size:** 30 seconds (50% overlap)
- **Features per sensor:** mean, std, min, max, range, skew, kurtosis
- **Total:** 5 sensors × 7 features = **35 features**

In [ ]:
from sklearn.model_selection import train_test_split

# Doğru yöntem: train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("=== GERÇEK TEST SONUCU ===")
print(classification_report(y_test, y_pred, target_names=['Baseline', 'Stres']))

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Baseline', 'Stres'],
            yticklabels=['Baseline', 'Stres'], ax=ax)
ax.set_title('Random Forest — Gerçek Test Sonucu')
ax.set_ylabel('Gerçek')
ax.set_xlabel('Tahmin')
plt.tight_layout()
plt.show()

In [ ]:
# Birden fazla katılımcı yükle
subjects = ['S2', 'S3', 'S4', 'S5', 'S6']
all_features = []

for subject in subjects:
    print(f"{subject} yükleniyor...")
    pkl_path = f'{wesad_path}/{subject}/{subject}.pkl'
    
    with open(pkl_path, 'rb') as f:
        data_s = pickle.load(f, encoding='latin1')
    
    labels_s = data_s['label']
    
    for start in range(0, len(labels_s) - WINDOW_SIZE, STEP_SIZE):
        end = start + WINDOW_SIZE
        label_window = labels_s[start:end]
        unique = np.unique(label_window)
        if len(unique) != 1 or unique[0] not in [1, 2]:
            continue
        
        window_features = {'subject': subject}
        for sensor in sensors:
            sig = data_s['signal']['chest'][sensor].flatten()[start:end]
            feats = extract_features(sig, label_window)
            for k, v in feats.items():
                if k != 'label':
                    window_features[f'{sensor}_{k}'] = v
        
        window_features['label'] = unique[0]
        all_features.append(window_features)

df_all = pd.DataFrame(all_features)
print(f"\n Toplam pencere: {len(df_all)}")
print(f"Label dağılımı:\n{df_all['label'].value_counts()}")
print(f"Katılımcı dağılımı:\n{df_all['subject'].value_counts()}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# X ve y ayır (subject kolonunu çıkar)
X = df_all.drop(['label', 'subject'], axis=1)
y = (df_all['label'] == 2).astype(int)  # 1=stres, 0=baseline

# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Modeller karşılaştır
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42)
}

print("=== MODEL KARŞILAŞTIRMASI ===\n")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    from sklearn.metrics import f1_score, accuracy_score
    print(f"{name}")
    print(f"  Accuracy : {accuracy_score(y_test, y_pred):.3f}")
    print(f"  F1 Score : {f1_score(y_test, y_pred):.3f}\n")

# Random Forest detaylı
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("=== RANDOM FOREST DETAYLI ===")
print(classification_report(y_test, y_pred, target_names=['Baseline', 'Stres']))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Baseline', 'Stres'],
            yticklabels=['Baseline', 'Stres'], ax=ax)
ax.set_title('Random Forest — 5 Katılımcı Test Sonucu')
ax.set_ylabel('Gerçek')
ax.set_xlabel('Tahmin')
plt.tight_layout()
plt.savefig('confusion_matrix_5subjects.png', dpi=150)
plt.show()

## 4. Machine Learning Models
We train and compare Random Forest and SVM classifiers.
An 80/20 train-test split is used to ensure honest evaluation and avoid overfitting.

In [ ]:
# SHAP yerine RF Feature Importance kullan
feature_names = X.columns.tolist()
importances = rf.feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Hangi Sensör Stresi En İyi Tahmin Ediyor?', fontsize=14, fontweight='bold')

# 1. Top 15 feature
sorted_idx = np.argsort(importances)[-15:]
axes[0].barh(range(15), importances[sorted_idx], color='#F44336', alpha=0.8)
axes[0].set_yticks(range(15))
axes[0].set_yticklabels([feature_names[i] for i in sorted_idx], fontsize=9)
axes[0].set_title('Top 15 En Önemli Feature')
axes[0].set_xlabel('Feature Importance')

# 2. Sensör bazında toplam önem
sensor_importance = {}
for sensor in sensors:
    cols = [i for i, f in enumerate(feature_names) if f.startswith(sensor)]
    sensor_importance[sensor] = importances[cols].sum()

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
bars = axes[1].bar(sensor_importance.keys(), sensor_importance.values(),
                   color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('Sensör Bazında Toplam Önem')
axes[1].set_ylabel('Toplam Importance')
for i, (k, v) in enumerate(sensor_importance.items()):
    axes[1].text(i, v + 0.002, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print(" Feature importance analizi tamamlandı!")

##  5. Feature Importance Analysis
We use Random Forest's built-in feature importance to identify
which sensors contribute most to stress detection.

##  Results & Discussion

### Model Performance
| Model | Accuracy | F1 Score |
|-------|----------|----------|
| Random Forest | 100% | 1.000 |
| SVM | 96.5% | 0.947 |

### Sensor Importance
- **EDA (0.336):** Strongest predictor — skin conductance directly reflects stress response
- **Resp (0.315):** Breathing pattern changes significantly under stress
- **ECG (0.173):** Heart rate variability provides meaningful contribution
- **EMG (0.110):** High inter-subject variability reduces predictive power
- **Temp (0.065):** Slow-changing signal carries less short-term information

### Limitations
- Only 5 out of 15 subjects used
- Lab-controlled environment may not reflect real-world conditions
- Inter-subject variability not addressed

### Future Work
- Apply Leave-One-Subject-Out (LOSO) cross-validation
- Extend to all 15 subjects
- Compare with deep learning approaches (LSTM, CNN)